# 02 — Silver Layer Cleaning

**Medallion Tier 2.** Parse & normalise the Bronze data, run the **config-driven data-quality engine**,
and split each source into a **clean** and a **quarantine** Delta table — then **MERGE-upsert** the clean
output so re-runs are idempotent.

| Output | Content |
|---|---|
| `silver_sales` / `silver_sales_quarantine` | cleaned transactions / rejected rows + reason |
| `silver_exchange_rate` / `silver_exchange_rate_quarantine` | validated rates / rejected rows |
| `audit_dq_results` | per-rule failure counts for this run |
| `audit_pipeline_run_log` | run metadata (rows in/out/quarantined, duration, status) |

All DQ rules (REJECT → quarantine, WARN → kept + flagged) come from `config/pipeline_config.json`.

In [ ]:
batch_id = ""  # passed in by the orchestration pipeline; generated if empty

In [ ]:
%run 00_common_utils

In [ ]:
batch_id = batch_id or new_batch_id()
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
log(f"Silver batch_id = {batch_id}")

ITEMS_SCHEMA = T.ArrayType(T.StructType([
    T.StructField("product_id", T.StringType()),
    T.StructField("category", T.StringType()),
    T.StructField("price", T.DoubleType()),
    T.StructField("quantity", T.IntegerType()),
]))
CUST_SCHEMA = T.StructType([
    T.StructField("name", T.StringType()),
    T.StructField("email", T.StringType()),
    T.StructField("phone", T.StringType()),
])

## 1. Exchange rate

In [ ]:
cfg_fx = CONFIG["silver"]["exchange_rate"]
rules_fx = CONFIG["data_quality"]["exchange_rate"]
run = start_run("silver", "exchange_rate", batch_id)
try:
    raw = spark.read.parquet(CONFIG["sources"]["exchange_rate"]["bronze_path"])
    stage = (raw
        .withColumn("year_num", F.col("year").cast("int"))
        .withColumn("month_num", F.col("month").cast("int"))
        .withColumn("rate_num", F.col("exchange_rate").cast("double"))
        .withColumn("from_cur", F.upper(F.trim("from_currency")))
        .withColumn("to_cur", F.upper(F.trim("to_currency"))))
    ev = apply_dq(stage, rules_fx)
    log_dq_results(ev, rules_fx, "exchange_rate", run)

    clean = (ev.where("_is_clean")
        .select(F.col("year_num").alias("year"), F.col("month_num").alias("month"),
                F.col("from_cur").alias("from_currency"), F.col("to_cur").alias("to_currency"),
                F.col("rate_num").alias("exchange_rate"))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_silver_loaded_at", F.current_timestamp()))
    quarantine = (ev.where("NOT _is_clean")
        .select("year", "month", "from_currency", "to_currency", "exchange_rate",
                "quarantine_reason", "_dq_failed_reject")
        .withColumn("_batch_id", F.lit(batch_id)))

    merge_upsert(clean, cfg_fx["clean_table"], cfg_fx["merge_key"])
    quarantine.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(cfg_fx["quarantine_table"])
    end_run(run, "SUCCESS", raw.count(), clean.count(), quarantine.count())
except Exception as e:
    end_run(run, "FAILED", message=str(e)); raise

## 2. Sales — parse, normalise & derive the columns the DQ rules check

In [ ]:
cfg_s = CONFIG["silver"]["sales"]
rules_s = CONFIG["data_quality"]["sales"]
run = start_run("silver", "sales", batch_id)
raw = spark.read.parquet(CONFIG["sources"]["sales"]["bronze_path"])
raw_count = raw.count()

order_ts = F.coalesce(
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"),
    F.to_timestamp("order_date", "yyyy-MM-dd'T'HH:mm:ss"),
    F.to_timestamp("order_date", "dd/MM/yyyy HH:mm"),
    F.to_timestamp("order_date", "dd/MM/yyyy"))

pm = F.lower(F.trim("payment_method"))
payment_norm = (F.when(pm.isin("credit_card", "creditcard", "credit card"), "Credit Card")
                 .when(pm == "paypal", "PayPal").when(pm == "cod", "COD")
                 .otherwise(F.initcap(pm)))

dup_win = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").asc_nulls_last(), F.col("order_date"))

stage = (raw
    .withColumn("order_ts", order_ts)
    .withColumn("total_amount_num", F.regexp_replace(F.col("total_amount"), "[^0-9.\\-]", "").cast("double"))
    .withColumn("shipping_cost_num", F.col("shipping_cost").cast("double"))
    .withColumn("items_parsed", F.from_json("items", ITEMS_SCHEMA))
    .withColumn("items_size", F.coalesce(F.size("items_parsed"), F.lit(0)))
    .withColumn("customer", F.from_json("customer_info", CUST_SCHEMA))
    .withColumn("feedback_num", F.when(F.trim("feedback_score") == "", None).otherwise(F.col("feedback_score").cast("double")))
    .withColumn("customer_age_num", F.when(F.trim("customer_age") == "", None).otherwise(F.col("customer_age").cast("int")))
    .withColumn("device_type_norm", F.when(F.trim("device_type") == "", "Unknown").otherwise(F.trim("device_type")))
    .withColumn("payment_method_norm", payment_norm)
    .withColumn("is_duplicate", F.row_number().over(dup_win) > 1))

## 3. Run the DQ engine, build the normalised schema, write clean + quarantine

In [ ]:
try:
    ev = apply_dq(stage, rules_s)
    log_dq_results(ev, rules_s, "sales", run, total=raw_count)

    clean = (ev.where("_is_clean").select(
        F.col("order_id"),
        F.col("order_ts").alias("order_date"),
        F.year("order_ts").alias("order_year"),
        F.month("order_ts").alias("order_month"),
        F.col("customer.name").alias("customer_name"),
        F.col("customer.email").alias("customer_email"),
        F.col("customer.phone").alias("customer_phone"),
        F.col("customer_age_num").alias("customer_age"),
        F.col("location"),
        F.col("device_type_norm").alias("device_type"),
        F.col("items_parsed").alias("items"),
        F.col("payment_method_norm").alias("payment_method"),
        F.upper(F.trim("currency")).alias("currency"),
        F.when(F.trim("discount_code") == "", None).otherwise(F.trim("discount_code")).alias("discount_code"),
        (F.trim("discount_code") != "").alias("has_discount"),
        F.col("shipping_cost_num").alias("shipping_cost"),
        F.col("total_amount_num").alias("total_amount"),
        F.col("order_status"),
        F.col("loyalty_points").cast("int").alias("loyalty_points"),
        F.col("feedback_num").alias("feedback_score"),
        F.col("feedback_num").between(1, 5).alias("feedback_is_valid"),
        F.col("_dq_failed_warn").alias("dq_warnings"))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_silver_loaded_at", F.current_timestamp()))

    quarantine = (ev.where("NOT _is_clean").select(
        "order_id", "order_date", "total_amount", "shipping_cost", "items",
        "payment_method", "order_status", "feedback_score",
        "quarantine_reason", "_dq_failed_reject")
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_silver_loaded_at", F.current_timestamp()))

    merge_upsert(clean, cfg_s["clean_table"], cfg_s["merge_key"], partition_by=cfg_s["partition_by"])
    quarantine.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(cfg_s["quarantine_table"])
    end_run(run, "SUCCESS", raw_count, clean.count(), quarantine.count())
except Exception as e:
    end_run(run, "FAILED", message=str(e)); raise

## 4. Data-quality report

In [ ]:
print("=== silver_sales ===", spark.table(cfg_s["clean_table"]).count(), "rows")
print("=== quarantine reasons ===")
spark.table(cfg_s["quarantine_table"]).groupBy("quarantine_reason").count().orderBy(F.desc("count")).show(truncate=False)
print("=== DQ results (this batch) ===")
(spark.table(AUDIT_DQ).where(F.col("batch_id") == batch_id)
    .select("entity", "rule_id", "severity", "reason", "failed_count", "total_count")
    .orderBy("entity", "rule_id").show(50, truncate=False))

In [ ]:
import notebookutils
notebookutils.notebook.exit(batch_id)